# Stream Per-gene Mean and Variance

This tutorial introduces single-pass streaming over an Atlas expression matrix.
The example computes the mean and variance of each selected gene without
materializing the complete cell-by-gene matrix in memory.

This is the simplest advanced example of using scAtlasPy as a data-access layer
for custom computation. It also introduces the distinction between
single-pass streams for complete summaries and multi-pass streams for iterative
model training.

By the end of this tutorial, you will be able to:

- define the cells, genes, and expression field used by a streaming
  calculation;
- inspect dense expression minibatches;
- estimate the memory required for a minibatch;
- calculate per-gene means and variances in one pass;
- validate and interpret the resulting statistics.

## Before You Begin

This tutorial assumes that:

- an Atlas has already been created;
- quality control and preprocessing have been completed;
- the expression field intended for the calculation is available;
- the selected dense minibatches fit in memory.

Open the existing Atlas:



In [ ]:
import os
from pathlib import Path
import numpy as np
import scatlaspy as sap

os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")

atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)



## 1. Define the Streaming View

Streaming routines operate on the current read index. Build the read index
before requesting minibatches:

In [ ]:
atlas.build_read_index(
    cell_condition=None,
    gene_condition=None,
    use_hvg=True,
    use_data="data_scale"
)



The read index defines the data traversed by the stream:

| Option | Meaning |
|---|---|
| `cell_condition` | Selects the cells included in the stream |
| `gene_condition` | Selects the genes eligible for inclusion |
| `use_hvg` | Further restricts the feature set to highly variable genes |
| `use_data` | Selects the expression field returned in each minibatch |

In this example, the stream contains:

- cells selected by `filter_cells`;
- genes selected by `filter_genes`;
- genes marked as highly variable;
- scaled expression values stored in `data_scale`.

Use `data_log1p` instead when the calculation should describe
log-normalized expression values.

```{important}
The output vectors follow the exact feature order defined by the current read
index. Record this gene order together with the calculated statistics.
```

## 2. Inspect the Minibatches

Inspect a few minibatches before implementing the complete calculation:



In [ ]:
for batch_id, X_batch in enumerate(
    atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=2048,
    ),
    start=1,
):
    print(
        f"Batch {batch_id}: "
        f"shape={X_batch.shape}, "
        f"dtype={X_batch.dtype}"
    )

    if batch_id == 3:
        break



A dense minibatch has shape:

```text
cells in the minibatch × selected genes
```

The final minibatch may contain fewer cells than the requested `batch_size`.

`pass_mode="single-pass"` traverses every cell in the current read index once.
The order of cells does not affect mean and variance calculations.

```{note}
Each call to `atlas.get_minibatch_dense()` creates a new stream. Stopping the
inspection loop above does not affect the new stream created for the complete
calculation below.
```

## 3. Choose a Batch Size

The memory required for one dense array is approximately:

```text
batch_size × n_genes × bytes_per_value
```

For 2,000 genes and a batch size of 2,048:

| Data type | Size of one dense batch |
|---|---:|
| `float32` | approximately 15.6 MiB |
| `float64` | approximately 31.3 MiB |

The peak memory used by the calculation is larger than the size of one batch.
The example below converts each minibatch to `float64` and creates a centered
copy, so multiple batch-sized arrays may briefly coexist.

Larger batches may improve throughput by reducing per-batch overhead, but they
also increase peak memory. Select a size that is appropriate for the number of
genes and the available memory.

### Limit the stream during development

Use `max_batches` to test code on a limited number of minibatches:



In [ ]:
for X_batch in atlas.get_minibatch_dense(
    pass_mode="single-pass",
    batch_size=2048,
    max_batches=20,
):
    pass



```{warning}
A stream limited by `max_batches` does not describe the complete read index.
Do not use a truncated stream for final whole-Atlas means or variances unless
the intended analysis is explicitly based on that subset.
```

## 4. Understand the Batch-wise Update

The calculation maintains three objects:

- `n_total`: the number of cells processed so far;
- `mean`: the running mean of each gene;
- `m2`: the running sum of squared deviations from the mean.

For each minibatch, the batch-level statistics are merged with the accumulated
statistics. If the existing data contain \(n_a\) cells and the new batch
contains \(n_b\) cells, the merged mean is:

```{math}
\mu_{a \cup b}
=
\mu_a
+
\frac{n_b}{n_a+n_b}
(\mu_b-\mu_a).
```

The sum of squared deviations is updated with:

```{math}
M_{2,a \cup b}
=
M_{2,a}
+
M_{2,b}
+
\frac{n_a n_b}{n_a+n_b}
(\mu_b-\mu_a)^2.
```

This is a batch-wise merge form of Welford's online algorithm. It is more
numerically stable than accumulating raw sums and squared sums separately.

## 5. Compute Mean and Variance

Initialize the running statistics:



In [ ]:
n_total = 0
mean = None
m2 = None
n_features = None



Traverse the selected expression matrix once:



In [ ]:
for batch_id, X_batch in enumerate(
    atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=2048,
    ),
    start=1,
):
    X_batch = np.asarray(X_batch, dtype=np.float64)

    if X_batch.ndim != 2:
        raise ValueError(
            "Each expression minibatch must be a two-dimensional matrix."
        )

    n_batch, batch_features = X_batch.shape

    if n_batch == 0:
        continue

    if n_features is None:
        n_features = batch_features
    elif batch_features != n_features:
        raise ValueError(
            "The number of features changed between minibatches."
        )

    batch_mean = X_batch.mean(axis=0)
    centered = X_batch - batch_mean

    # Sum of squared deviations for each gene without creating
    # another complete squared minibatch.
    batch_m2 = np.einsum(
        "ij,ij->j",
        centered,
        centered,
        optimize=True,
    )

    if n_total == 0:
        n_total = n_batch
        mean = batch_mean.copy()
        m2 = batch_m2
    else:
        delta = batch_mean - mean
        new_total = n_total + n_batch

        mean += delta * (n_batch / new_total)
        m2 += (
            batch_m2
            + delta * delta * n_total * n_batch / new_total
        )

        n_total = new_total

    if batch_id == 1 or batch_id % 100 == 0:
        print(f"Processed {n_total:,} cells")



Finalize the sample variance and standard deviation:



In [ ]:
if n_total == 0:
    raise ValueError(
        "The current read index did not yield any cells."
    )

if n_total < 2:
    raise ValueError(
        "At least two cells are required to calculate sample variance."
    )

variance = m2 / (n_total - 1)
std = np.sqrt(variance)

print(
    f"Processed {n_total:,} cells "
    f"and {mean.shape[0]:,} genes"
)

print(
    f"Mean range: "
    f"[{mean.min():.4f}, {mean.max():.4f}]"
)

print(
    f"Variance range: "
    f"[{variance.min():.4f}, {variance.max():.4f}]"
)

print(
    f"Standard-deviation range: "
    f"[{std.min():.4f}, {std.max():.4f}]"
)



The denominator `n_total - 1` produces the sample variance. Use `n_total`
instead when the required statistic is the population variance.

```{note}
The calculation uses `float64` accumulation even when the stored minibatches
use `float32`. The additional precision reduces accumulated numerical error
during long streaming calculations.
```

## 6. Validate the Result

Check that the output dimensions are consistent:



In [ ]:
if mean.ndim != 1:
    raise ValueError("The mean output is not a vector.")

if variance.shape != mean.shape:
    raise ValueError(
        "The mean and variance vectors have different shapes."
    )

if std.shape != mean.shape:
    raise ValueError(
        "The mean and standard-deviation vectors have different shapes."
    )



Check for non-finite values:



In [ ]:
if not np.isfinite(mean).all():
    raise ValueError("The mean vector contains non-finite values.")

if not np.isfinite(variance).all():
    raise ValueError("The variance vector contains non-finite values.")

if not np.isfinite(std).all():
    raise ValueError(
        "The standard-deviation vector contains non-finite values."
    )



Variance should be nonnegative:



In [ ]:
minimum_variance = variance.min()
print(f"Minimum variance: {minimum_variance:.6e}")



A substantial negative value indicates a calculation or data problem. Very
small negative values can occasionally arise from floating-point roundoff in
other implementations.

## 7. Interpret the Result

### Scaled expression

When the read index uses:



In [ ]:
use_data="data_scale"



gene means may be close to 0 and standard deviations may be close to 1 when:

- scaling was calculated using the same cells;
- the same genes are selected;
- the scaled values were not clipped;
- the scaling and variance calculations use compatible denominator
  conventions.

The statistics may differ from 0 and 1 when:

- the current read index selects a subset of the cells used for scaling;
- scaled values were clipped;
- the selected genes differ from those used during scaling;
- the scaling function and this calculation use different degrees of freedom;
- a gene had zero or very low variance.

Large deviations therefore do not necessarily indicate an implementation
error. Interpret them in relation to how the scaled representation was created.

### Log-normalized expression

When the read index uses:



In [ ]:
use_data="data_log1p"



the result describes the mean and variance of the stored log-transformed
expression values. These values should not be expected to have z-score
properties.

The mean of `log1p` expression is also not equivalent to applying `log1p` to
the mean count value.

## 8. Compare with an In-memory Reference

When developing a custom streaming statistic, validate it on a dataset or
subset that fits in memory.

For the small PBMC3K tutorial Atlas, collect the selected read-index matrix in
memory and calculate the corresponding NumPy statistics:



In [ ]:
X_reference = np.vstack([
    np.asarray(batch, dtype=np.float64)
    for batch in atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=2048,
    )
])

reference_mean = X_reference.mean(axis=0)
reference_variance = X_reference.var(
    axis=0,
    ddof=1,
)



Compare the results only when the streaming and reference calculations use:

- exactly the same cells;
- exactly the same genes;
- exactly the same gene order;
- the same expression representation;
- the same variance denominator.



In [ ]:
np.testing.assert_allclose(
    mean,
    reference_mean,
    rtol=1e-7,
    atol=1e-9,
)

np.testing.assert_allclose(
    variance,
    reference_variance,
    rtol=1e-6,
    atol=1e-8,
)


## Single-pass and Multi-pass Streams

This tutorial uses `pass_mode="single-pass"` because a complete summary should
visit every selected cell once.

Use a multi-pass stream when fitting an iterative method that needs randomized
and repeated access to the data. Examples include neural networks, stochastic
optimization, and minibatch clustering.

| Stream mode | Typical purpose |
|---|---|
| `single-pass` | Complete statistics, inference, export, and deterministic traversal |
| `multi-pass` | Iterative training with repeated randomized minibatches |



## Close the Atlas

Close the database connection when this tutorial is complete. This releases
the DuckDB file lock so the same `.sasql` Atlas can be opened by another
notebook or Python session.


In [ ]:
atlas.close()


## Next Steps

Continue with {doc}`stream-covariance-matrix` to extend the same batch-wise
merge pattern from independent per-gene summaries to a gene-by-gene covariance
matrix.

See {doc}`implement-minibatch-kmeans` for an iterative method that uses
randomized, multi-pass minibatches.